In [2]:
import pandas as pd
import itertools

transactions = [
    ['Bread', 'Milk'],
    ['Bread', 'Diaper', 'Beer', 'Eggs'],
    ['Milk', 'Diaper', 'Beer', 'Coke'],
    ['Bread', 'Milk', 'Diaper', 'Beer'],
    ['Bread', 'Milk', 'Diaper', 'Coke']
]

def get_itemsets(transactions, length):
    items = set()
    for t in transactions:
        for item in itertools.combinations(sorted(set(t)), length):
            items.add(item)
    return list(items)

def get_frequent_itemsets(itemsets, transactions, min_support):
    freq_itemsets = {}
    for item in itemsets:
        count = sum(1 for t in transactions if set(item).issubset(set(t)))
        support = count / len(transactions)
        if support >= min_support:
            freq_itemsets[item] = support
    return freq_itemsets

def apriori(transactions, min_support):
    result = []
    k = 1
    while True:
        itemsets = get_itemsets(transactions, k)
        freq_itemsets = get_frequent_itemsets(itemsets, transactions, min_support)
        if not freq_itemsets:
            break
        result.extend([(item, support) for item, support in freq_itemsets.items()])
        transactions = [t for t in transactions if all(set(item).issubset(set(t)) for item in freq_itemsets)]
        k += 1
    return result

frequent_itemsets = apriori(transactions, min_support=0.6)
df = pd.DataFrame(frequent_itemsets, columns=['Itemset', 'Support'])
print(df)

                        Itemset  Support
0                     (Diaper,)      0.8
1                       (Beer,)      0.6
2                      (Bread,)      0.8
3                       (Milk,)      0.8
4               (Bread, Diaper)      1.0
5                 (Beer, Bread)      1.0
6                  (Beer, Milk)      1.0
7                 (Bread, Milk)      1.0
8                (Diaper, Milk)      1.0
9                (Beer, Diaper)      1.0
10         (Beer, Diaper, Milk)      1.0
11        (Beer, Bread, Diaper)      1.0
12          (Beer, Bread, Milk)      1.0
13        (Bread, Diaper, Milk)      1.0
14  (Beer, Bread, Diaper, Milk)      1.0


In [3]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)
frequent_itemsets = apriori(df, min_support=0.6, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.7)

print("Frequent Itemsets:\n", frequent_itemsets)
print("\nAssociation Rules:\n", rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

Frequent Itemsets:
    support         itemsets
0      0.6           (Beer)
1      0.8          (Bread)
2      0.8         (Diaper)
3      0.8           (Milk)
4      0.6   (Diaper, Beer)
5      0.6  (Bread, Diaper)
6      0.6    (Bread, Milk)
7      0.6   (Diaper, Milk)

Association Rules:
   antecedents consequents  support  confidence    lift
0    (Diaper)      (Beer)      0.6        0.75  1.2500
1      (Beer)    (Diaper)      0.6        1.00  1.2500
2     (Bread)    (Diaper)      0.6        0.75  0.9375
3    (Diaper)     (Bread)      0.6        0.75  0.9375
4     (Bread)      (Milk)      0.6        0.75  0.9375
5      (Milk)     (Bread)      0.6        0.75  0.9375
6    (Diaper)      (Milk)      0.6        0.75  0.9375
7      (Milk)    (Diaper)      0.6        0.75  0.9375


In [4]:
import pandas as pd

data = {
    'Antecedent': ['A', 'A', 'A', 'A', 'B', 'B', 'B'],
    'Consequent': [0, 0, 1, 0, 1, 0, 1]
}

df = pd.DataFrame(data)

total_rows = len(df)

support_consequent = df['Consequent'].value_counts().to_dict()

antecedent_counts = df['Antecedent'].value_counts().to_dict()

grouped = df.groupby(['Antecedent', 'Consequent']).size().reset_index(name='count')

results = []

for _, row in grouped.iterrows():
    antecedent = row['Antecedent']
    consequent = row['Consequent']
    count = row['count']
    
    support = count / total_rows
    confidence = count / antecedent_counts[antecedent]
    lift = confidence / (support_consequent[consequent] / total_rows)
    
    results.append({
        'Rule': f"{antecedent} -> {consequent}",
        'Support': round(support, 4),
        'Confidence': round(confidence, 4),
        'Lift': round(lift, 4)
    })

result_df = pd.DataFrame(results)
print(result_df)

     Rule  Support  Confidence    Lift
0  A -> 0   0.4286      0.7500  1.3125
1  A -> 1   0.1429      0.2500  0.5833
2  B -> 0   0.1429      0.3333  0.5833
3  B -> 1   0.2857      0.6667  1.5556
